# 3 — Modeling

This notebook trains and compares three models:
1. **Hourly Baseline** — historical average by (day_of_week, hour)
2. **LightGBM Spike Classifier** — binary spike detection
3. **LightGBM Price Regressor** — LMP level prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pjm_spike_forecast.data import build_demo_dataset
from pjm_spike_forecast.features import build_feature_matrix, get_feature_columns
from pjm_spike_forecast.models import HourlyBaseline, SpikeClassifier, PriceRegressor
from pjm_spike_forecast.evaluation import classification_metrics, regression_metrics

df = build_demo_dataset(n_days=365, seed=42)
feat_df = build_feature_matrix(df)
feature_cols = get_feature_columns(feat_df)

# 80/20 temporal split
split = int(len(feat_df) * 0.8)
train = feat_df.iloc[:split]
test = feat_df.iloc[split:]
X_train, X_test = train[feature_cols], test[feature_cols]
print(f"Train: {len(train)} rows ({train.index.min().date()} → {train.index.max().date()})")
print(f"Test:  {len(test)} rows ({test.index.min().date()} → {test.index.max().date()})")

## 3.1 — Hourly Baseline

In [ ]:
baseline = HourlyBaseline()
baseline.fit(train)

bl_price = baseline.predict_price(test)
bl_spike = baseline.predict_spike(test)

bl_reg = regression_metrics(test["lmp"].values, bl_price)
bl_clf = classification_metrics(test["spike"].values, bl_spike)

print("Baseline Price Metrics:")
for k, v in bl_reg.items():
    print(f"  {k}: {v:.4f}")
print("\nBaseline Spike Metrics:")
for k, v in bl_clf.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## 3.2 — LightGBM Spike Classifier

In [ ]:
clf = SpikeClassifier()
clf.fit(X_train, train["spike"])

spike_pred = clf.predict(X_test)
spike_proba = clf.predict_proba(X_test)

clf_metrics = classification_metrics(test["spike"].values, spike_pred, spike_proba)
print("LightGBM Spike Classifier:")
for k, v in clf_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## 3.3 — LightGBM Price Regressor

In [ ]:
reg = PriceRegressor()
reg.fit(X_train, train["lmp"])

price_pred = reg.predict(X_test)
reg_metrics = regression_metrics(test["lmp"].values, price_pred)
print("LightGBM Price Regressor:")
for k, v in reg_metrics.items():
    print(f"  {k}: {v:.4f}")

## 3.4 — Side-by-Side Comparison

In [ ]:
comparison = pd.DataFrame({
    "Metric": ["RMSE", "MAE", "R²"],
    "Baseline": [bl_reg["rmse"], bl_reg["mae"], bl_reg["r2"]],
    "LightGBM": [reg_metrics["rmse"], reg_metrics["mae"], reg_metrics["r2"]],
})
print(comparison.to_string(index=False))

print(f"\nLightGBM RMSE improvement over baseline: "
      f"{(1 - reg_metrics['rmse'] / bl_reg['rmse']) * 100:.1f}%")

## 3.5 — Feature Importance (SHAP)

In [ ]:
from pjm_spike_forecast.evaluation import compute_shap_importance
from pjm_spike_forecast.visualization import plot_feature_importance

shap_df = compute_shap_importance(clf, X_test, max_samples=500)
fig = plot_feature_importance(shap_df, top_n=15)
plt.show()

print("\nTop 10 features by SHAP importance:")
print(shap_df.head(10).to_string(index=False))